# Data Preparation

In [ ]:
# Data Preparation for COVID-19 Image Analysis
# This notebook prepares the dataset for training

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set up paths
DATA_DIR = Path('data')
TRAIN_CSV = DATA_DIR / 'train.csv'
META_TRAIN_CSV = DATA_DIR / 'meta_train.csv'
META_TEST_CSV = DATA_DIR / 'meta_test.csv'
TRAIN_IMAGES_DIR = DATA_DIR / '256px' / 'train' / 'train'
TEST_IMAGES_DIR = DATA_DIR / '256px' / 'test' / 'test'

print("Data directories:")
print(f"Train images: {TRAIN_IMAGES_DIR}")
print(f"Test images: {TEST_IMAGES_DIR}")
print(f"Train CSV: {TRAIN_CSV.exists()}")
print(f"Meta train CSV: {META_TRAIN_CSV.exists()}")
print(f"Meta test CSV: {META_TEST_CSV.exists()}")

In [ ]:
# Load the training data
print("Loading training data...")
train_df = pd.read_csv(TRAIN_CSV)
meta_train_df = pd.read_csv(META_TRAIN_CSV)
meta_test_df = pd.read_csv(META_TEST_CSV)

print(f"\nTrain CSV shape: {train_df.shape}")
print(f"Meta train CSV shape: {meta_train_df.shape}")
print(f"Meta test CSV shape: {meta_test_df.shape}")

print("\nFirst few rows of train.csv:")
print(train_df.head())


In [ ]:
# Explore the data structure
print("Dataset Information:")
print(f"\nColumns in train.csv: {train_df.columns.tolist()}")
print(f"\nUnique labels: {train_df['label_id'].unique()}")
print(f"\nLabel distribution:")
print(train_df['label_id'].value_counts())
print(f"\nStudy label distribution:")
print(train_df['study_label'].value_counts().sort_index())


In [ ]:
# IMPORTANT: Image files are named with pattern: {StudyInstanceUID}_{something}_{ImageInstanceUID}.jpg
# So we need to search for files ending with _{ImageInstanceUID}.jpg
# This cell provides the corrected function to find image paths

from glob import glob

def get_image_path_fixed(image_id, is_train=True):
    """Find the image file path by searching for files ending with _{ImageInstanceUID}.jpg"""
    if is_train:
        search_dir = TRAIN_IMAGES_DIR
    else:
        search_dir = TEST_IMAGES_DIR
    
    # Search for files ending with _{image_id}.jpg
    pattern = str(search_dir / f"*_{image_id}.jpg")
    matches = glob(pattern)
    
    if matches:
        return matches[0]  # Return the first match
    else:
        return None  # Return None if not found

# Test the function with a known image ID
test_id = "64a776818efe"  # From your example
test_path = get_image_path_fixed(test_id, is_train=True)
if test_path:
    print(f"✓ Found test image: {os.path.basename(test_path)}")
else:
    print(f"⚠ Test image not found (may not be downloaded yet)")
    print(f"  Looking for pattern: *_{test_id}.jpg")


In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
train_df['label_id'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of COVID-19 Appearance Types', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Label', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].tick_params(axis='x', rotation=45)

# Study label distribution
train_df['study_label'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribution of Study Labels (Numeric)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Study Label', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)

plt.tight_layout()
plt.show()

# Print statistics
print("\nClass Distribution Statistics:")
print(train_df['label_id'].value_counts())
print(f"\nTotal images: {len(train_df)}")
print(f"Images with bounding boxes: {train_df[train_df['boxes'].str.contains('width', na=False)].shape[0]}")


In [ ]:
# Check if image files exist and create file paths
# Using the fixed function that searches for files ending with _{ImageInstanceUID}.jpg

# Add image paths to dataframe using the fixed function
print("Finding image file paths...")
train_df['image_path'] = train_df['ImageInstanceUID'].apply(lambda x: get_image_path_fixed(x, is_train=True))

# Check how many images actually exist (path will be None if not found)
train_df['image_exists'] = train_df['image_path'].notna()

# Print debug info for a few
print("\nSample image paths found:")
sample_paths = train_df[train_df['image_exists']]['image_path'].head(3).tolist()
if sample_paths:
    for path in sample_paths:
        if path:
            print(f"  {os.path.basename(path)}")
else:
    print("  No images found yet - make sure images are downloaded to the data folder")

print(f"\nImages found: {train_df['image_exists'].sum()} out of {len(train_df)}")
print(f"Missing images: {(~train_df['image_exists']).sum()}")

# Filter to only include images that exist
train_df = train_df[train_df['image_exists']].copy()
print(f"\nFinal dataset size: {len(train_df)} images")


In [ ]:
# Create train/validation split
# Using stratified split to maintain class distribution
X = train_df[['ImageInstanceUID', 'image_path']]
y = train_df['study_label']  # Using numeric labels for stratification

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Create train and validation dataframes
train_split_df = train_df[train_df['ImageInstanceUID'].isin(X_train['ImageInstanceUID'])].copy()
val_split_df = train_df[train_df['ImageInstanceUID'].isin(X_val['ImageInstanceUID'])].copy()

print(f"Training set: {len(train_split_df)} images")
print(f"Validation set: {len(val_split_df)} images")
print(f"\nTraining set class distribution:")
print(train_split_df['label_id'].value_counts())
print(f"\nValidation set class distribution:")
print(val_split_df['label_id'].value_counts())


In [ ]:
# Visualize sample images from each class
import cv2

def display_sample_images(df, num_samples=4):
    """Display sample images from each class"""
    fig, axes = plt.subplots(len(df['label_id'].unique()), num_samples, 
                            figsize=(15, 3 * len(df['label_id'].unique())))
    
    if len(df['label_id'].unique()) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, label in enumerate(sorted(df['label_id'].unique())):
        label_samples = df[df['label_id'] == label].sample(min(num_samples, len(df[df['label_id'] == label])))
        
        for col, (_, row) in enumerate(label_samples.iterrows()):
            img_path = row['image_path']
            if img_path and os.path.exists(str(img_path)):
                img = cv2.imread(str(img_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[idx, col].imshow(img)
                axes[idx, col].set_title(f"{label}\n{row['ImageInstanceUID'][:8]}", fontsize=9)
                axes[idx, col].axis('off')
            else:
                axes[idx, col].text(0.5, 0.5, 'Image not found', ha='center', va='center')
                axes[idx, col].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Sample images from training set:")
display_sample_images(train_split_df, num_samples=3)


In [ ]:
# Save prepared data splits
train_split_df.to_csv(DATA_DIR / 'train_split.csv', index=False)
val_split_df.to_csv(DATA_DIR / 'val_split.csv', index=False)

print("Saved data splits:")
print(f"  - {DATA_DIR / 'train_split.csv'}")
print(f"  - {DATA_DIR / 'val_split.csv'}")

# Create a summary
summary = {
    'total_images': len(train_df),
    'train_images': len(train_split_df),
    'val_images': len(val_split_df),
    'classes': train_df['label_id'].unique().tolist(),
    'class_distribution': train_df['label_id'].value_counts().to_dict()
}

print("\nData Preparation Summary:")
print(f"  Total images: {summary['total_images']}")
print(f"  Training images: {summary['train_images']}")
print(f"  Validation images: {summary['val_images']}")
print(f"  Classes: {summary['classes']}")
print("\nData preparation complete! Ready for model training.")
